# Autonomous Remediation

Close the loop — the Supervisor delegates remediation to an **Actions Agent** that executes approved fixes.

## Safety Model — Defense in Depth

```
┌─────────────────────────────────────────────────────────┐
│  Layer 1: Bedrock Guardrail                             │
│  Blocks dangerous SQL at the LLM level                  │
├─────────────────────────────────────────────────────────┤
│  Layer 2: Risk Classification                           │
│  LOW = auto-approved │ MEDIUM/HIGH = needs approval     │
├─────────────────────────────────────────────────────────┤
│  Layer 3: Human Approval (Email)                        │
│  HTML email with Approve/Reject buttons                 │
├─────────────────────────────────────────────────────────┤
│  Layer 4: Code-Level Validation                         │
│  SQL parsing rejects dangerous operations               │
└─────────────────────────────────────────────────────────┘
```

---
## Step 1: Create Bedrock Guardrail

Blocks dangerous SQL (DROP, DELETE, TRUNCATE), prompt injection, and off-topic requests at the model level.

In [ ]:
%%bash
cd /workshop/labs/remediation
python3 setup_guardrail.py

In [ ]:
# Source the environment variables created by the script
import os

env_file = os.path.expanduser('~/.dbops_env')
if os.path.exists(env_file):
    with open(env_file) as f:
        for line in f:
            if line.startswith('export '):
                key, val = line.replace('export ', '').strip().split('=', 1)
                os.environ[key] = val

print(f"✅ BEDROCK_GUARDRAIL_ID: {os.getenv('BEDROCK_GUARDRAIL_ID')}")
print(f"✅ BEDROCK_GUARDRAIL_VERSION: {os.getenv('BEDROCK_GUARDRAIL_VERSION')}")

---
## Step 2: Create Approval Workflow

Creates DynamoDB table + Lambda + API Gateway + SES for email-based human-in-the-loop approval.

⚠️ **You'll be prompted for your email address.** Enter it and check your inbox to verify.

In [ ]:
%%bash
cd /workshop/labs/remediation
python3 setup_approval_workflow.py

In [ ]:
# Reload environment variables
env_file = os.path.expanduser('~/.dbops_env')
if os.path.exists(env_file):
    with open(env_file) as f:
        for line in f:
            if line.startswith('export '):
                key, val = line.replace('export ', '').strip().split('=', 1)
                os.environ[key] = val

print(f"✅ APPROVAL_API_URL: {os.getenv('APPROVAL_API_URL', 'NOT SET')}")
print(f"✅ SES_SENDER_EMAIL: {os.getenv('SES_SENDER_EMAIL', 'NOT SET')}")

---
## Step 3: Diagnose and Resolve

The Supervisor now has the Actions Agent as a 6th sub-agent. One prompt triggers the full flow:

**Diagnosis → Recommendation → Approval Email → Execution**

Run the Supervisor and ask it to fix the performance issue:

In [ ]:
# Start the Supervisor with Actions Agent enabled
# Type: "The database CPU is high. Diagnose the root cause and fix it."
# Then check your email for the approval request!
!python3 /workshop/labs/interactive_agents/supervisor_agent_interactive.py

---
## How It Works

```
You: "Fix the high CPU"
       ↓
Supervisor → Query Performance Agent (diagnoses slow queries)
       ↓
Supervisor → Actions Agent (proposes CREATE INDEX)
       ↓
Actions Agent → classify_action_risk() → MEDIUM
       ↓
Actions Agent → request_human_approval() → sends email
       ↓
You click "Approve" in email
       ↓
Actions Agent → create_index() → executes on SQL Server
       ↓
Actions Agent → send_email_notification() → audit trail
```

**Next:** Deploy the MCP Gateway and connect to AWS DevOps Agent.